# MetroPT-3 Benchmark: Exploratory Data Analysis & Leakage-Free Preprocessing

**Author:** Herath H.M.N.P ()
**Component:** 02 — Predictive Maintenance & RUL Forecasting
**Project:** J26-SE-326 (Industrial Air Compressor Predictive Maintenance)

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add root to sys.path
REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from c02_predictive_maintenance_rul.src.preprocessing.data_cleaner import DataCleaner, ANALOG_SENSOR_COLS
from c02_predictive_maintenance_rul.src.preprocessing.sequence_window_generator import SequenceWindowGenerator

print("Modules imported successfully.")

In [ ]:
cleaner = DataCleaner()
df_raw = cleaner.load_and_sort("../data/sample_metropt3_snippet.csv")
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()

In [ ]:
df_clean = cleaner.handle_missing_and_outliers(df_raw)
df_clean[ANALOG_SENSOR_COLS].describe().round(3)

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df_clean[ANALOG_SENSOR_COLS].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("MetroPT-3 Analog Sensor Cross-Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
generator = SequenceWindowGenerator(window_size=60, step_size=1)
train_df, val_df, test_df = generator.chronological_split(df_clean, train_ratio=0.70, val_ratio=0.15)

# Leakage-free StandardScaler fitted ONLY on train
cleaner.fit_scaler(train_df)
train_scaled = cleaner.transform_features(train_df)
test_scaled = cleaner.transform_features(test_df)
print("Verification: Chronological Split complete without data leakage!")